In [2]:
import os
import re
import html
from pathlib import Path
from collections import Counter

import pandas as pd
from tqdm import tqdm

# =========================================================
# 1. USER SETTINGS
# =========================================================

CORE_AI_DICT_FILE = r"/Volumes/ORICO/Dissertation/Hospitality_Core_AI_Dictionary_v8_LOCK.xlsx"
PREAI_DICT_FILE   = r"/Volumes/ORICO/Dissertation/Hospitality_PreAI_Tech_Dictionary_v7_LOCK.xlsx"

CORE_AI_SHEET = "Dictionary"
PREAI_SHEET   = "Dictionary"

NEWSROOM_FOLDER = r"/Volumes/ORICO/Dissertation/News data cleaned"

DOC_OUTPUT_FILE       = r"/Volumes/ORICO/Dissertation/newsroom_doc_level_results.xlsx"
FIRMYEAR_OUTPUT_FILE  = r"/Volumes/ORICO/Dissertation/newsroom_firm_year_results.xlsx"
TERMCOUNT_OUTPUT_FILE = r"/Volumes/ORICO/Dissertation/newsroom_term_counts.xlsx"
DIAG_OUTPUT_FILE      = r"/Volumes/ORICO/Dissertation/newsroom_diagnostics.xlsx"

SAVE_TERM_COUNTS = True
MIN_WORDS = 20


# =========================================================
# 2. LOAD LOCKED DICTIONARIES
# =========================================================

def load_terms(dict_file, sheet_name="Dictionary", term_col_candidates=("Terms", "Term", "Keyword", "Keywords")):
    """
    Load dictionary terms from Excel.
    Automatically identifies the header row containing the term column.
    """
    raw = pd.read_excel(dict_file, sheet_name=sheet_name, header=None)

    header_row = None
    candidates_lower = [c.lower() for c in term_col_candidates]

    for i in range(min(10, len(raw))):
        row_values = [str(x).strip().lower() for x in raw.iloc[i].tolist() if pd.notna(x)]
        if any(v in candidates_lower for v in row_values):
            header_row = i
            break

    if header_row is None:
        raise ValueError(
            f"Could not find a header row with one of {term_col_candidates} "
            f"in the first 10 rows of sheet '{sheet_name}'."
        )

    df = pd.read_excel(dict_file, sheet_name=sheet_name, header=header_row)

    term_col = None
    for c in df.columns:
        if str(c).strip().lower() in candidates_lower:
            term_col = c
            break

    if term_col is None:
        raise ValueError(f"Could not find a term column in {df.columns.tolist()}")

    terms = (
        df[term_col]
        .dropna()
        .astype(str)
        .str.strip()
        .tolist()
    )

    terms = [t for t in terms if t]
    terms = list(dict.fromkeys(terms))
    return terms


# =========================================================
# 3. TEXT NORMALIZATION
# =========================================================

def clean_text(text):
    if pd.isna(text):
        return ""

    text = str(text)
    text = html.unescape(text)

    # remove HTML tags
    text = re.sub(r"<[^>]+>", " ", text)

    # normalize whitespace
    text = text.replace("\xa0", " ")
    text = re.sub(r"[\r\n\t]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def normalize_term(term):
    term = html.unescape(term)
    term = term.lower().strip()
    term = term.replace("-", " ")
    term = term.replace("_", " ")
    term = term.replace("/", " ")
    term = term.replace(".", "")
    term = re.sub(r"\s+", " ", term).strip()
    return term


def word_count(text):
    if not text:
        return 0
    return len(re.findall(r"\b\w+\b", text))


# =========================================================
# 4. DICTIONARY PATTERN BUILDING
# =========================================================

def build_term_pattern(term):
    """
    Build a regex for one finalized dictionary term.
    Preserves the dictionary content while allowing flexible spacing.
    """
    term_norm = normalize_term(term)
    escaped = re.escape(term_norm)
    escaped = escaped.replace(r"\ ", r"\s+")
    pattern = rf"(?<!\w){escaped}(?!\w)"
    return re.compile(pattern, flags=re.IGNORECASE)


def count_terms(text, compiled_patterns):
    """
    Returns:
        total_hits: total number of matched occurrences
        matched_counter: Counter of matched dictionary terms
    """
    total_hits = 0
    matched_counter = Counter()

    if not text:
        return total_hits, matched_counter

    for term, pattern in compiled_patterns.items():
        matches = pattern.findall(text)
        if matches:
            n = len(matches)
            total_hits += n
            matched_counter[term] += n

    return total_hits, matched_counter


# =========================================================
# 5. FILE FILTERING
# =========================================================

def is_valid_input_file(path: Path) -> bool:
    """
    Exclude Mac metadata files and temporary Excel lock files.
    """
    name = path.name

    if name.startswith("._"):
        return False

    if name.startswith("~$"):
        return False

    if name.startswith("."):
        return False

    return True


# =========================================================
# 6. COLUMN STANDARDIZATION
# =========================================================

def normalize_col(col):
    if pd.isna(col):
        return ""
    col = str(col).strip().lower()
    col = re.sub(r"\s+", "", col)
    return col


def standardize_newsroom_columns(df):
    """
    Supports Korean and English newsroom headers.
    """
    rename_map = {}

    for c in df.columns:
        cn = normalize_col(c)

        if cn in ["제목", "title", "headline", "subject"]:
            rename_map[c] = "title"
        elif cn in ["본문", "텍스트", "내용", "text", "body", "content", "article"]:
            rename_map[c] = "body"
        elif cn in ["시간", "날짜", "date", "posted", "published", "publicationdate", "publishdate"]:
            rename_map[c] = "date_raw"
        elif cn in ["링크", "url", "link", "제목_링크", "제목링크"]:
            rename_map[c] = "link"
        elif cn in ["회사", "firm", "company", "corp", "issuer", "brand"]:
            rename_map[c] = "firm"
        else:
            rename_map[c] = c

    df = df.rename(columns=rename_map)

    for col in ["title", "body", "date_raw", "link", "firm"]:
        if col not in df.columns:
            df[col] = None

    return df


# =========================================================
# 7. DATE PARSING
# =========================================================

def parse_date_and_year(x):
    """
    Returns:
        parsed_date : pd.Timestamp or NaT
        year        : int or None
        date_type   : full_date / year_only / missing / unparsed
    """
    if pd.isna(x) or str(x).strip() == "":
        return pd.NaT, None, "missing"

    raw = str(x).strip()

    # year only
    if re.fullmatch(r"\d{4}", raw):
        return pd.NaT, int(raw), "year_only"

    if re.fullmatch(r"\d{4}\.0", raw):
        return pd.NaT, int(float(raw)), "year_only"

    # general parse
    parsed = pd.to_datetime(raw, errors="coerce")
    if pd.notna(parsed):
        return parsed, int(parsed.year), "full_date"

    # explicit formats
    fmts = [
        "%m.%d.%y", "%m.%d.%Y",
        "%Y.%m.%d",
        "%m/%d/%y", "%m/%d/%Y",
        "%A, %B %d, %Y",
        "%B %d, %Y",
    ]

    for fmt in fmts:
        try:
            parsed = pd.to_datetime(raw, format=fmt)
            return parsed, int(parsed.year), "full_date"
        except Exception:
            pass

    return pd.NaT, None, "unparsed"


def infer_firm_from_filename(file_path: Path):
    return file_path.stem.strip()


# =========================================================
# 8. TXT LOADING
# =========================================================

def read_text_file(file_path: Path):
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return file_path.read_text(encoding=enc, errors="ignore")
        except Exception:
            continue
    return ""


def parse_txt_date_from_text(text):
    """
    Try to infer date from raw TXT article content.
    """
    text = clean_text(text)

    # Example: December 26, 2024
    m = re.search(r"\b([A-Z][a-z]+ \d{1,2}, \d{4})\b", text)
    if m:
        parsed = pd.to_datetime(m.group(1), errors="coerce")
        if pd.notna(parsed):
            return parsed, int(parsed.year), "full_date"

    # fallback year
    m2 = re.search(r"\b(20\d{2})\b", text)
    if m2:
        return pd.NaT, int(m2.group(1)), "year_only"

    return pd.NaT, None, "missing"


# =========================================================
# 9. PROCESS NEWSROOM FILES
# =========================================================

def process_newsroom_xlsx(file_path: Path):
    df = pd.read_excel(file_path, engine="openpyxl")
    df = standardize_newsroom_columns(df)

    df["title"] = df["title"].apply(clean_text)
    df["body"] = df["body"].apply(clean_text)
    df["link"] = df["link"].apply(clean_text)

    # fill firm using filename when missing
    inferred_firm = infer_firm_from_filename(file_path)

    if df["firm"].isna().all() or (df["firm"].astype(str).str.strip() == "").all():
        df["firm"] = inferred_firm
    else:
        df["firm"] = df["firm"].fillna("").astype(str).str.strip()
        df.loc[df["firm"] == "", "firm"] = inferred_firm

    parsed = df["date_raw"].apply(parse_date_and_year)
    df["date"] = parsed.apply(lambda x: x[0])
    df["year"] = parsed.apply(lambda x: x[1])
    df["date_type"] = parsed.apply(lambda x: x[2])

    df["full_text"] = (df["title"].fillna("") + " " + df["body"].fillna("")).str.strip()
    df["full_text"] = df["full_text"].apply(clean_text)

    df["source_file"] = file_path.name
    df["source_type"] = "newsroom_xlsx"
    df["doc_word_count"] = df["full_text"].apply(word_count)

    # remove empty docs
    df = df[df["full_text"].str.len() > 0].copy()

    return df[[
        "firm", "date_raw", "date", "year", "date_type",
        "title", "body", "full_text", "link",
        "source_file", "source_type", "doc_word_count"
    ]]


def process_newsroom_txt(file_path: Path):
    raw_text = read_text_file(file_path)
    text = clean_text(raw_text)

    parsed_date, year, date_type = parse_txt_date_from_text(text)

    row = {
        "firm": infer_firm_from_filename(file_path),
        "date_raw": None,
        "date": parsed_date,
        "year": year,
        "date_type": date_type,
        "title": "",
        "body": text,
        "full_text": text,
        "link": "",
        "source_file": file_path.name,
        "source_type": "newsroom_txt",
        "doc_word_count": word_count(text)
    }

    return pd.DataFrame([row])


def load_newsroom_corpus(folder_path):
    folder = Path(folder_path)

    xlsx_files = sorted(list(folder.glob("*.xlsx")) + list(folder.glob("*.xls")))
    txt_files = sorted(list(folder.glob("*.txt")))

    # filter invalid files
    xlsx_files = [fp for fp in xlsx_files if is_valid_input_file(fp)]
    txt_files = [fp for fp in txt_files if is_valid_input_file(fp)]

    frames = []

    for fp in tqdm(xlsx_files, desc="Processing newsroom Excel files"):
        try:
            frames.append(process_newsroom_xlsx(fp))
        except Exception as e:
            print(f"[XLSX ERROR] {fp.name}: {e}")

    for fp in tqdm(txt_files, desc="Processing newsroom TXT files"):
        try:
            frames.append(process_newsroom_txt(fp))
        except Exception as e:
            print(f"[TXT ERROR] {fp.name}: {e}")

    if not frames:
        return pd.DataFrame()

    docs = pd.concat(frames, ignore_index=True)

    # deduplicate
    docs = docs.drop_duplicates(subset=["firm", "year", "title", "full_text"]).copy()

    # minimum word threshold
    docs = docs[docs["doc_word_count"] >= MIN_WORDS].copy()

    return docs


# =========================================================
# 10. APPLY FINALIZED DICTIONARIES
# =========================================================

def apply_dictionary_scoring(df, core_ai_patterns, preai_patterns):
    out = df.copy()

    core_hits_list = []
    pre_hits_list = []
    core_focus_list = []
    pre_focus_list = []

    core_term_counter = Counter()
    pre_term_counter = Counter()

    for text, n_words in tqdm(
        zip(out["full_text"], out["doc_word_count"]),
        total=len(out),
        desc="Scoring newsroom documents"
    ):
        text_norm = normalize_term(clean_text(text))

        core_hits, core_counter = count_terms(text_norm, core_ai_patterns)
        pre_hits, pre_counter = count_terms(text_norm, preai_patterns)

        core_hits_list.append(core_hits)
        pre_hits_list.append(pre_hits)

        if n_words > 0:
            core_focus_list.append(core_hits / n_words * 1_000_000)
            pre_focus_list.append(pre_hits / n_words * 1_000_000)
        else:
            core_focus_list.append(None)
            pre_focus_list.append(None)

        core_term_counter.update(core_counter)
        pre_term_counter.update(pre_counter)

    out["Core_AI_hits"] = core_hits_list
    out["PreAI_Tech_hits"] = pre_hits_list
    out["Core_AI_Focus_per_million_words"] = core_focus_list
    out["PreAI_Focus_per_million_words"] = pre_focus_list

    term_count_rows = []

    for term, cnt in core_term_counter.items():
        term_count_rows.append({
            "dictionary": "Core_AI",
            "term": term,
            "count": cnt
        })

    for term, cnt in pre_term_counter.items():
        term_count_rows.append({
            "dictionary": "PreAI_Tech",
            "term": term,
            "count": cnt
        })

    term_counts_df = pd.DataFrame(term_count_rows)

    if not term_counts_df.empty:
        term_counts_df = term_counts_df.sort_values(
            ["dictionary", "count"],
            ascending=[True, False]
        )

    return out, term_counts_df


# =========================================================
# 11. FIRM-YEAR AGGREGATION
# =========================================================

def aggregate_firm_year(df):
    temp = df.dropna(subset=["year"]).copy()
    temp["year"] = temp["year"].astype(int)

    agg = (
        temp.groupby(["firm", "year"], as_index=False)
        .agg(
            n_docs=("full_text", "count"),
            total_words=("doc_word_count", "sum"),
            total_core_ai_hits=("Core_AI_hits", "sum"),
            total_preai_hits=("PreAI_Tech_hits", "sum"),
            mean_doc_core_ai_focus=("Core_AI_Focus_per_million_words", "mean"),
            mean_doc_preai_focus=("PreAI_Focus_per_million_words", "mean")
        )
    )

    agg["Core_AI_Focus_per_million_words"] = (
        agg["total_core_ai_hits"] / agg["total_words"] * 1_000_000
    )

    agg["PreAI_Focus_per_million_words"] = (
        agg["total_preai_hits"] / agg["total_words"] * 1_000_000
    )

    return agg


# =========================================================
# 12. DIAGNOSTICS
# =========================================================

def build_diagnostics(docs, firm_year):
    rows = [
        {"metric": "n_documents", "value": len(docs)},
        {"metric": "n_firms", "value": docs["firm"].nunique()},
        {"metric": "n_firm_year_rows", "value": len(firm_year)},
        {"metric": "pct_full_date", "value": (docs["date_type"] == "full_date").mean()},
        {"metric": "pct_year_only", "value": (docs["date_type"] == "year_only").mean()},
        {"metric": "pct_missing_or_unparsed_date", "value": docs["date_type"].isin(["missing", "unparsed"]).mean()},
        {"metric": "mean_doc_words", "value": docs["doc_word_count"].mean()},
        {"metric": "median_doc_words", "value": docs["doc_word_count"].median()},
        {"metric": "mean_core_ai_focus_doc", "value": docs["Core_AI_Focus_per_million_words"].mean()},
        {"metric": "mean_preai_focus_doc", "value": docs["PreAI_Focus_per_million_words"].mean()},
    ]
    return pd.DataFrame(rows)


# =========================================================
# 13. MAIN
# =========================================================

def main():
    print("Loading finalized dictionaries...")
    core_ai_terms = load_terms(CORE_AI_DICT_FILE, sheet_name=CORE_AI_SHEET)
    preai_terms   = load_terms(PREAI_DICT_FILE, sheet_name=PREAI_SHEET)

    print(f"Core AI terms: {len(core_ai_terms)}")
    print(f"Pre-AI terms: {len(preai_terms)}")

    core_ai_patterns = {term: build_term_pattern(term) for term in core_ai_terms}
    preai_patterns   = {term: build_term_pattern(term) for term in preai_terms}

    print("Loading newsroom corpus...")
    docs = load_newsroom_corpus(NEWSROOM_FOLDER)

    if docs.empty:
        print("No newsroom documents found.")
        return

    print(f"Documents loaded: {len(docs):,}")

    print("Applying finalized dictionaries...")
    docs_scored, term_counts_df = apply_dictionary_scoring(
        docs,
        core_ai_patterns=core_ai_patterns,
        preai_patterns=preai_patterns
    )

    print("Aggregating to firm-year...")
    firm_year = aggregate_firm_year(docs_scored)

    print("Building diagnostics...")
    diagnostics = build_diagnostics(docs_scored, firm_year)

    print("Saving outputs...")

    with pd.ExcelWriter(DOC_OUTPUT_FILE, engine="openpyxl") as writer:
        docs_scored.to_excel(writer, sheet_name="doc_level", index=False)

    with pd.ExcelWriter(FIRMYEAR_OUTPUT_FILE, engine="openpyxl") as writer:
        firm_year.to_excel(writer, sheet_name="firm_year", index=False)

    if SAVE_TERM_COUNTS:
        with pd.ExcelWriter(TERMCOUNT_OUTPUT_FILE, engine="openpyxl") as writer:
            term_counts_df.to_excel(writer, sheet_name="term_counts", index=False)

    with pd.ExcelWriter(DIAG_OUTPUT_FILE, engine="openpyxl") as writer:
        diagnostics.to_excel(writer, sheet_name="diagnostics", index=False)

    print("Done.")
    print(f"Saved doc-level results to: {DOC_OUTPUT_FILE}")
    print(f"Saved firm-year results to: {FIRMYEAR_OUTPUT_FILE}")
    if SAVE_TERM_COUNTS:
        print(f"Saved term counts to: {TERMCOUNT_OUTPUT_FILE}")
    print(f"Saved diagnostics to: {DIAG_OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Loading finalized dictionaries...
Core AI terms: 61
Pre-AI terms: 57
Loading newsroom corpus...


/var/folders/9q/rg7dh5cj2mx7xmtkv28d6jm80000gn/T/ipykernel_1790/1461497901.py:247: FutureWarning: Parsing 'EDT' as tzlocal (dependent on system timezone) is deprecated and will raise in a future version. Pass the 'tz' keyword or call tz_localize after construction instead
  parsed = pd.to_datetime(raw, errors="coerce")
Processing newsroom Excel files:   9%|▉          | 8/93 [00:03<00:24,  3.43it/s]

[XLSX ERROR] BBX CAPITAL CORP.xlsx: 'NoneType' object has no attribute 'total_seconds'


Processing newsroom Excel files:  44%|████▍     | 41/93 [00:25<02:10,  2.51s/it]

[XLSX ERROR] INTERCONTINENTAL HOTELS GRP.xlsx: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().


...

[XLSX ERROR] RED ROBIN GOURMET BURGERS.xlsx: 'NoneType' object has no attribute 'total_seconds'


Processing newsroom TXT files: 100%|█████████████| 2/2 [00:00<00:00, 163.20it/s]


Documents loaded: 18,975
Applying finalized dictionaries...


Scoring newsroom documents: 100%|████████| 18975/18975 [02:05<00:00, 151.50it/s]


Aggregating to firm-year...
Building diagnostics...
Saving outputs...
Done.
Saved doc-level results to: /Volumes/ORICO/Dissertation/newsroom_doc_level_resultsAIRBNB_INC.xlsx
Saved firm-year results to: /Volumes/ORICO/Dissertation/newsroom_firm_year_resultsAIRBNB_INC.xlsx
Saved term counts to: /Volumes/ORICO/Dissertation/newsroom_term_countsAIRBNB_INC.xlsx
Saved diagnostics to: /Volumes/ORICO/Dissertation/newsroom_diagnosticsAIRBNB_INC.xlsx


In [1]:
import pandas as pd

file_path = r"/Volumes/ORICO/Dissertation/newsroom_firm_year_results.xlsx"

df = pd.read_excel(file_path)

print(df.shape)
df.head()

(588, 10)


,firm,year,n_docs,total_words,total_core_ai_hits,total_preai_hits,mean_doc_core_ai_focus,mean_doc_preai_focus,Core_AI_Focus_per_million_words,PreAI_Focus_per_million_words
0,AEGIS BRANDS INC,2019,3,1647,0,0,0.0,0.000000,0.0,0.000000
1,AEGIS BRANDS INC,2021,7,8016,0,0,0.0,0.000000,0.0,0.000000
2,AEGIS BRANDS INC,2022,3,8913,0,2,0.0,260.824204,0.0,224.391338
3,AEGIS BRANDS INC,2023,2,978,0,0,0.0,0.000000,0.0,0.000000
4,AEGIS BRANDS INC,2024,8,7479,0,1,0.0,126.903553,0.0,133.707715


In [2]:
df.sort_values(
    "Core_AI_Focus_per_million_words",
    ascending=False
).head(20)

,firm,year,n_docs,total_words,total_core_ai_hits,total_preai_hits,mean_doc_core_ai_focus,mean_doc_preai_focus,Core_AI_Focus_per_million_words,PreAI_Focus_per_million_words
308,MERITAGE HOSPITALITY GROUP,2021,1,390,3,0,7692.307692,0.000000,7692.307692,0.000000
29,BBQ HOLDINGS INC,2023,1,904,1,0,1106.194690,0.000000,1106.194690,0.000000
481,SODEXO S A,2019,32,13619,10,2,592.358817,127.811861,734.268302,146.853660
384,PIZZA PIZZA ROYALTY CORP,2022,3,1560,1,0,619.578686,0.000000,641.025641,0.000000
486,SODEXO S A,2024,31,22569,13,2,706.906846,111.182752,576.011343,88.617130
16,ARAMARK,2024,162,105977,50,8,480.670845,76.504407,471.800485,75.488078
545,WENDY'S CO,2023,51,39387,14,35,321.816886,956.515834,355.447229,888.618072
11,ARAMARK,2019,55,37403,13,1,276.623363,42.680324,347.565703,26.735823
161,DOMINO'S PIZZA INC,2023,31,28876,8,5,326.663944,291.271586,277.046682,173.154176
13,ARAMARK,2021,41,25661,7,2,255.836688,44.502207,272.787499,77.939285
